# Statistics and Experiments: Intervals, A/B Tests and Forecasts

Notebook for **Chapter 5** of the *ways* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/ways/ch05.html

Tip: change a number, predict the output, then run the cell again.

In [ ]:
# Setup: the files used in this chapter are downloaded next to the notebook.
import os, urllib.request
os.makedirs("data", exist_ok=True)
for f in ["customers.csv", "orders.csv", "products.csv", "shopkart.db"]:
    if not os.path.exists("data/" + f):
        urllib.request.urlretrieve("https://dadipavan.github.io/Learning/study/ways/data/" + f, "data/" + f)


## 5.1 Estimating a number with uncertainty

The average delivery time in our data is a **statistic**: a number computed from a sample. If ShopKart had a different but similar year, we would get a slightly different average. A **confidence interval** (CI) gives a range of plausible values for the true long-run average. For a mean, the 95 percent CI is

In [ ]:
DATA = "data/"
import sys
sys.path.insert(0, DATA)
import prep
import numpy as np
from scipy import stats

o, c, p = prep.load_clean(DATA)
x = o.delivery_minutes.dropna().values
n, mean, s = len(x), x.mean(), x.std(ddof=1)

se = s / np.sqrt(n)
t = stats.t.ppf(0.975, n - 1)
print(f"n={n} mean={mean:.3f} sd={s:.3f} se={se:.4f} t={t:.4f}")
print("by hand :", round(mean - t * se, 3), round(mean + t * se, 3))
print("scipy   :", [round(float(v), 3) for v in stats.t.interval(0.95, n - 1, loc=mean, scale=se)])

rng = np.random.default_rng(42)
boot = np.array([rng.choice(x, n).mean() for _ in range(2000)])
print("bootstrap:", [round(float(v), 3) for v in np.percentile(boot, [2.5, 97.5])])

n=17208 mean=40.221 sd=9.092 se=0.0693 t=1.9601
by hand : 40.085 40.357
scipy   : [40.085, 40.357]
bootstrap: [40.084, 40.358]


## 5.2 Proportions: the late rate and the Web versus App question

Many business metrics are proportions: late rate, return rate, conversion. For a proportion p̂ = k/n the standard error is √(p̂(1 − p̂)/n). To compare two groups, use the difference p̂1 − p̂2 and the **two-proportion z-test**: pool both groups under the idea "there is no difference", get z = difference ÷ pooled standard error, and convert z into a **p-value**, the probability, if there were no true difference, of seeing a gap at least this large by chance.

In [ ]:
DATA = "data/"
import sys
sys.path.insert(0, DATA)
import prep
import numpy as np
from scipy import stats

o, c, p = prep.load_clean(DATA)
r = o.dropna(subset=["late"])
a = r[r.channel == "App"].late.values
w = r[r.channel == "Web"].late.values
n1, n2, k1, k2 = len(w), len(a), w.sum(), a.sum()
p1, p2 = k1 / n1, k2 / n2
pbar = (k1 + k2) / (n1 + n2)
z = (p1 - p2) / np.sqrt(pbar * (1 - pbar) * (1 / n1 + 1 / n2))
print(f"Web {p1:.4f} (n={n1})  App {p2:.4f} (n={n2})  diff={p1 - p2:.4f}")
print(f"by hand : z={z:.3f}  p={2 * (1 - stats.norm.cdf(abs(z))):.5f}")

table = [[k1, n1 - k1], [k2, n2 - k2]]
chi2, pv, _, _ = stats.chi2_contingency(table, correction=False)
print(f"scipy   : chi2={chi2:.3f} (= z^2 {z ** 2:.3f})  p={pv:.5f}")

rng = np.random.default_rng(1)
both = np.concatenate([w, a])
count = 0
for _ in range(5000):
    rng.shuffle(both)
    d = both[:n1].mean() - both[n1:].mean()
    count += abs(d) >= abs(p1 - p2)
print("permutation p ~", (count + 1) / 5001)

diffs = [rng.choice(w, n1).mean() - rng.choice(a, n2).mean() for _ in range(3000)]
print("bootstrap 95% CI for the difference:", [round(float(v), 4) for v in np.percentile(diffs, [2.5, 97.5])])

Web 0.1490 (n=6643)  App 0.1305 (n=10514)  diff=0.0185
by hand : z=3.433  p=0.00060
scipy   : chi2=11.782 (= z^2 11.782)  p=0.00060
permutation p ~ 0.0005998800239952009
bootstrap 95% CI for the difference: [0.0076, 0.0293]


## 5.3 Significant is not the same as explained: confounding

The data generator did *not* build any channel effect into delivery times, so we know the truth here, which real analysts never do. Why does the test see a gap anyway? There are two honest possibilities: a chance result (with p = 0.0006 that would be unusual, but we have run many comparisons in this document), or a feature we did not plan. First check the obvious suspect. Lateness depends on the promise (45 or 60 minutes), and a channel that happened to receive more 45-minute promises would look worse for a reason unrelated to the channel. A variable that is unevenly spread across groups and drives the outcome is a **confounder**. The remedy is to compare like with like: **stratify** (compare within each promise level) and combine.

In [ ]:
DATA = "data/"
import sys
sys.path.insert(0, DATA)
import prep
o, c, p = prep.load_clean(DATA)
r = o.dropna(subset=["late"])
r = r[r.channel.isin(["App", "Web"])]
t = r.groupby(["promised_minutes", "channel"]).late.agg(["mean", "count"]).round(4)
print(t.to_string())
share = r.groupby("channel").promised_minutes.apply(lambda s: (s == 45).mean()).round(4)
print("share of 45-minute promises:", share.to_dict())

# standardise: weight each stratum by its overall size
w = r.promised_minutes.value_counts(normalize=True)
adj = r.groupby(["channel", "promised_minutes"]).late.mean().unstack()
print("adjusted late rate:", (adj * w).sum(axis=1).round(4).to_dict())

mean  count
promised_minutes channel               
45.0             App      0.2397   5140
                 Web      0.2662   3332
60.0             App      0.0261   5374
                 Web      0.0311   3311
share of 45-minute promises: {'App': 0.4889, 'Web': 0.5016}
adjusted late rate: {'App': 0.1315, 'Web': 0.1472}


## 5.4 Designing an A/B test: sample size and the peeking trap

Before launching a test you choose three things: the **baseline rate** (say a late rate of 13.8 percent), the **minimum detectable effect** (the smallest improvement worth acting on, say 1 percentage point), and the error rates (5 percent false alarms, 80 percent power: a real effect of that size is detected 80 percent of the time). The number of customers per group is approximately

In [ ]:
DATA = "data/"
import numpy as np
from scipy import stats

p0, delta = 0.138, 0.01
za, zb = stats.norm.ppf(0.975), stats.norm.ppf(0.80)
n = int(np.ceil(2 * (za + zb) ** 2 * p0 * (1 - p0) / delta ** 2))
print("formula: n per group =", n)

rng = np.random.default_rng(3)
def detects(n, p_a, p_b):
    a, b = rng.binomial(n, p_a), rng.binomial(n, p_b)
    pbar = (a + b) / (2 * n)
    z = (a / n - b / n) / np.sqrt(2 * pbar * (1 - pbar) / n)
    return abs(z) > 1.96
power = np.mean([detects(n, p0, p0 - delta) for _ in range(4000)])
fa = np.mean([detects(n, p0, p0) for _ in range(4000)])
print(f"simulated power with a real 1-point effect: {power:.3f}")
print(f"simulated false alarm rate with no effect:  {fa:.3f}")

formula: n per group = 18674
simulated power with a real 1-point effect: 0.819
simulated false alarm rate with no effect:  0.052


In [ ]:
DATA = "data/"
import numpy as np
rng = np.random.default_rng(11)
looks, per_look, p = 20, 500, 0.138
stopped = 0
runs = 2000
for _ in range(runs):
    a = rng.binomial(1, p, looks * per_look)
    b = rng.binomial(1, p, looks * per_look)
    for i in range(1, looks + 1):
        n = i * per_look
        pa, pb = a[:n].mean(), b[:n].mean()
        pbar = (pa + pb) / 2
        z = (pa - pb) / np.sqrt(2 * pbar * (1 - pbar) / n)
        if abs(z) > 1.96:
            stopped += 1
            break
print("share of no-effect experiments declared significant when peeking 20 times:", round(stopped / runs, 3))

share of no-effect experiments declared significant when peeking 20 times: 0.235


## 5.5 Comparing means: do discounts increase units?

Q9 asked whether discounted orders carry more units. This is a comparison of two means. The **Welch t-test** compares them without assuming equal variances. By hand: t = (x̄1 − x̄2) ÷ √(s1²/n1 + s2²/n2).

In [ ]:
DATA = "data/"
import sys
sys.path.insert(0, DATA)
import prep
import numpy as np
from scipy import stats
o, c, p = prep.load_clean(DATA)
a = o[o.discount > 0].quantity.values
b = o[o.discount == 0].quantity.values
se = np.sqrt(a.var(ddof=1) / len(a) + b.var(ddof=1) / len(b))
t = (a.mean() - b.mean()) / se
print(f"discounted {a.mean():.4f} (n={len(a)})  full price {b.mean():.4f} (n={len(b)})")
print(f"by hand: diff={a.mean() - b.mean():.4f} t={t:.3f}")
res = stats.ttest_ind(a, b, equal_var=False)
print(f"scipy  : t={res.statistic:.3f} p={res.pvalue:.3f}")
lo, hi = (a.mean() - b.mean()) + np.array([-1.96, 1.96]) * se
print(f"95% CI for the difference: {lo:.4f} to {hi:.4f}")

discounted 1.9542 (n=11386)  full price 1.9405 (n=8609)
by hand: diff=0.0137 t=0.928
scipy  : t=0.928 p=0.353
95% CI for the difference: -0.0152 to 0.0427


## 5.6 Forecasting: five methods, one hold-out

A forecast is a statement about the future that will be checked. Judging it honestly needs a **hold-out**: train on the past, hide the most recent weeks, predict them, and compare the prediction with what happened. Never evaluate a forecast on the data used to fit it. We forecast daily order counts for the last 28 days of the year, using only data before 4 December.

In [ ]:
DATA = "data/"
import sys
sys.path.insert(0, DATA)
import prep
import numpy as np, pandas as pd

o, c, p = prep.load_clean(DATA)
y = o.groupby("order_date").size().astype(float)
H = 28
tr, te = y.iloc[:-H], y.iloc[-H:]
f = {}
f["naive"] = np.repeat(tr.iloc[-1], H)
f["seasonal naive (7)"] = np.tile(tr.iloc[-7:].values, 4)
f["moving average 7"] = np.repeat(tr.iloc[-7:].mean(), H)

def level(x, a):
    l = x[0]
    for v in x[1:]:
        l = a * v + (1 - a) * l
    return l
def sse(a):
    l, e = tr.iloc[0], 0.0
    for v in tr.iloc[1:]:
        e += (v - l) ** 2
        l = a * v + (1 - a) * l
    return e
alphas = np.linspace(0.05, 0.95, 19)
alpha = alphas[np.argmin([sse(a) for a in alphas])]
f["exp. smoothing"] = np.repeat(level(tr.values, alpha), H)

def design(idx, t0):
    t = np.arange(len(idx)) + t0
    return np.column_stack([np.ones(len(idx)), t] + [(idx.dayofweek == d).astype(float) for d in range(1, 7)])
beta = np.linalg.lstsq(design(tr.index, 0), tr.values, rcond=None)[0]
f["linear trend + weekday"] = design(te.index, len(tr)) @ beta

print("alpha chosen on training data:", round(alpha, 2))
rows = []
for k, v in f.items():
    e = te.values - v
    rows.append((k, np.abs(e).mean(), np.sqrt((e ** 2).mean()), 100 * np.abs(e / te.values).mean(), e.mean()))
print(pd.DataFrame(rows, columns=["method", "MAE", "RMSE", "MAPE%", "bias"]).round(2).sort_values("MAE").to_string(index=False))
print("mean actual in hold-out:", round(te.mean(), 2), "| mean of last 28 training days:", round(tr.iloc[-28:].mean(), 2))

alpha chosen on training data: 0.15
                method   MAE  RMSE  MAPE%   bias
                 naive  9.82 12.37  24.51  -8.25
linear trend + weekday 19.05 20.45  44.26 -18.62
        exp. smoothing 22.09 23.94  52.22 -22.09
      moving average 7 26.68 28.22  62.21 -26.68
    seasonal naive (7) 27.75 32.86  63.17 -26.68
mean actual in hold-out: 47.75 | mean of last 28 training days: 63.46


In [ ]:
DATA = "data/"
import sys
sys.path.insert(0, DATA)
import prep
import numpy as np

o, c, p = prep.load_clean(DATA)
y = o.groupby("order_date").size().astype(float)
h = 14
scores = {"naive": [], "moving average 7": [], "seasonal naive (7)": []}
for origin in range(200, 340, 28):
    tr, te = y.iloc[:origin], y.iloc[origin:origin + h]
    pred = {"naive": np.repeat(tr.iloc[-1], h),
            "moving average 7": np.repeat(tr.iloc[-7:].mean(), h),
            "seasonal naive (7)": np.tile(tr.iloc[-7:].values, 2)}
    for k, v in pred.items():
        scores[k].append(np.abs(te.values - v).mean())
for k, v in scores.items():
    print(f"{k:20s} mean MAE over {len(v)} origins: {np.mean(v):6.2f}   per origin: {np.round(v, 1)}")

naive                mean MAE over 5 origins:  10.86   per origin: [ 9.  11.1 11.7 12.4 10.1]
moving average 7     mean MAE over 5 origins:   9.39   per origin: [ 9.6 11.6  8.4  7.  10.4]
seasonal naive (7)   mean MAE over 5 origins:  10.34   per origin: [11.1  9.  10.9  8.6 12.1]


## 5.7 Choosing the tool for statistics

## Chapter recap

## Practice set

Problem 5.8.1 A sample of 400 deliveries has mean 40 and standard deviation 9. Give a 95 percent CI for the mean (use 1.96).

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/ways/ch05.html